## Chinese Tokenization : jieba
## English: NLTK, spaCy, Hugging Face Tokenizers (for BERT ..)


jieba 的默认模式实现的就是这种方式。它会先基于一个前缀词典（Trie 树），高效地构建出一个包含句子中所有可能词语组合的有向无环图（DAG）。接着，通过动态规划算法寻找一条概率最大的路径，作为最终分词结果。这个过程可以被量化为一个概率计算问题。

Log 概率与动态规划, 在实际工程中，将大量小于 1 的概率值直接相乘，很容易导致结果趋近于 0，造成浮点数下溢，无法比较路径优劣。为此，jieba 采用了对数概率技术，利用对数函数 log 的性质，将概率的累乘转换为 log 概率的累加。寻找概率最大值就等价于寻找 log 概率之和的最大值，有效避免了下溢问题.解决了数值计算的稳定性问题后，
为了避免暴力计算所有可能路径带来的巨大计算量，jieba 还引入了动态规划的思想。它会从句子的末尾开始，从后向前递推计算到每个位置的最优切分路径及其 log 概率之和，并记录下来。最终，从句子开头出发，根据记录好的最优路径信息，就能反推出整个句子的最优分词结果。

未登录词（OOV, Out-Of-Vocabulary）是一个“相对概念”——相对某个词典/词表未被收录的词。对词典法来说，就是词典里没有。典型表现是本该作为一个整体的词因未被收录而被错误地切碎成单字或无关的片段。

In [2]:
import jieba
text = "我在梦里收到清华大学录取通知书"
seg_list = jieba.lcut(text, cut_all=False)
print(seg_list)

E:\anaconda\envs\pytorchp\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\mingw\AppData\Local\Temp\jieba.cache
Loading model cost 0.982 seconds.
Prefix dict has been built successfully.


['我', '在', '梦里', '收到', '清华大学', '录取', '通知书']


In [2]:
dag = jieba.get_DAG(text)
print(dag)

{0: [0], 1: [1], 2: [2], 3: [3], 4: [4, 5], 5: [5], 6: [6, 7, 9], 7: [7, 8], 8: [8, 9], 9: [9], 10: [10, 11], 11: [11], 12: [12, 13, 14], 13: [13], 14: [14]}


In [3]:
from transformers import AutoTokenizer

E:\anaconda\envs\pytorchp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

In [5]:
text = "我在梦里收到清华大学录取通知书"

tokens = tokenizer(text)
print(tokens["input_ids"])

[101, 2769, 1762, 3457, 7027, 3119, 1168, 3926, 1290, 1920, 2110, 2497, 1357, 6858, 4761, 741, 102]


In [6]:
decoded = tokenizer.decode(tokens["input_ids"])
print(decoded)

[CLS] 我 在 梦 里 收 到 清 华 大 学 录 取 通 知 书 [SEP]


In [10]:
# jieba version
text = "我在Boss直聘找工作"

seg_list_hmm = jieba.lcut(text, HMM=True)
print("hmm", seg_list_hmm)

hmm ['我', '在', 'Boss', '直聘', '找', '工作']


In [2]:
import jieba
text = "我在Boss直聘找工作"

seg_list_hmm = jieba.lcut(text, HMM=False)
print("hmm", seg_list_hmm)

E:\anaconda\envs\pytorchp\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\mingw\AppData\Local\Temp\jieba.cache
Loading model cost 1.356 seconds.
Prefix dict has been built successfully.


hmm ['我', '在', 'Boss', '直', '聘', '找', '工作']


In [3]:
import spacy
nlp = spacy.load("zh_core_web_sm")
doc = nlp(text)
print([token.text for token in doc])

E:\anaconda\envs\pytorchp\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


['我', '在', 'Boss', '直聘', '找', '工作']


In [2]:
import jieba
import jieba.posseg as pseg
text = "九头虫让奔波儿灞把唐僧师徒除掉"

words = pseg.lcut(text, HMM=False)
print(words)

E:\anaconda\envs\pytorchp\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\mingw\AppData\Local\Temp\jieba.cache
Loading model cost 1.549 seconds.
Prefix dict has been built successfully.


[pair('九头', 'm'), pair('虫', 'n'), pair('让', 'v'), pair('奔波', 'v'), pair('儿', 'n'), pair('灞', 'x'), pair('把', 'p'), pair('唐僧', 'nr'), pair('师徒', 'n'), pair('除掉', 'v')]


In [3]:
jieba.load_userdict("./t.txt")
dic_words = pseg.lcut(text, HMM=False)
print(dic_words)

[pair('九', 'n'), pair('头', 'n'), pair('虫', 'n'), pair('让', 'v'), pair('奔波儿灞', 'nr'), pair('把', 'p'), pair('唐僧', 'nr'), pair('师徒', 'n'), pair('除掉', 'v')]
